# 多个.jsonl文件合并

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List, Dict, Any, Tuple, Union


JsonlItem = Union[str, Tuple[str, str]]  # path 或 (name, path)


def merge_jsonl_files(
    jsonl_items: List[JsonlItem],
    out_path: str,
    source_name_field: str = "source_file",           # 原本就有：来源名字
    source_path_field: str = "source_file_path",      # ✅新增：来源文件路径
    id_joiner: str = "__",                            # ✅拼接 source_file和旧id 的分隔符
) -> str:
    """
    合并多个 .jsonl 为一个统一 schema 的 .jsonl：
      - 不改动原有字段的值(除了 id 会被重写）
      - 缺失字段补 None（写出为 JSON null）
      - ✅新增：每行记录来源 source_name_field（例如 source_file）
      - ✅新增：每行记录来源文件路径 source_path_field（例如 source_file_path）
      - ✅把每行的 id 改成：source_name + id_joiner + 原id
      - ✅若输入文件中已存在 source_name_field / source_path_field 字段，则直接报错（避免覆盖）
      - 打印：每个输入文件写出多少行 + 合并后总行数

    jsonl_items 支持两种输入：
      1) list[str]                  -> 只能得到文件路径，source_name 自动用文件名
      2) list[(name, path)]         -> 你手动指定 name（推荐）
    """
    if not jsonl_items:
        raise ValueError("jsonl_items 不能为空")

    # 解析成 [(name, Path)]
    items: List[Tuple[str, Path]] = []
    for x in jsonl_items:
        if isinstance(x, (tuple, list)) and len(x) == 2:
            name, p = str(x[0]), Path(x[1])
        elif isinstance(x, str):
            p = Path(x)
            name = p.name
        else:
            raise TypeError("jsonl_items 里的元素必须是 str 或 (name, path)")

        if not p.exists():
            raise FileNotFoundError(f"输入文件不存在: {p}")
        items.append((name, p))

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # 1) 第一遍：收集全局字段（按首次出现顺序）
    global_keys: List[str] = []
    global_key_set = set()

    for _, p in items:
        with p.open("r", encoding="utf-8") as f:
            for lineno, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception as e:
                    raise ValueError(f"JSON 解析失败: file={p} line={lineno} err={e}") from e
                if not isinstance(obj, dict):
                    raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno} got={type(obj)}")

                # ✅新增：检测字段冲突（避免覆盖）
                if source_name_field in obj:
                    raise KeyError(
                        f"字段冲突: 文件 {p} 第 {lineno} 行已存在字段 '{source_name_field}'，"
                        f"合并时该字段会被覆盖，请先改名或删除该字段后再合并。"
                    )
                if source_path_field in obj:
                    raise KeyError(
                        f"字段冲突: 文件 {p} 第 {lineno} 行已存在字段 '{source_path_field}'，"
                        f"合并时该字段会被覆盖，请先改名或删除该字段后再合并。"
                    )

                for k in obj.keys():
                    if k not in global_key_set:
                        global_key_set.add(k)
                        global_keys.append(k)

    # ✅确保来源字段存在（插到最前面，方便查看）
    prepend_fields = []
    for f in [source_name_field, source_path_field]:
        if f not in global_key_set:
            prepend_fields.append(f)
            global_key_set.add(f)

    if prepend_fields:
        global_keys = prepend_fields + global_keys

    # 2) 第二遍：写出并计数（总数 + 每文件）
    written_total = 0
    per_file_written: Dict[str, int] = {str(p): 0 for _, p in items}

    with out_path.open("w", encoding="utf-8") as out_f:
        for name, p in items:
            with p.open("r", encoding="utf-8") as f:
                for lineno, line in enumerate(f, start=1):
                    raw = line.strip()
                    if not raw:
                        continue
                    try:
                        obj = json.loads(raw)
                    except Exception as e:
                        raise ValueError(f"JSON 解析失败: file={p} line={lineno} err={e}") from e
                    if not isinstance(obj, dict):
                        raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno} got={type(obj)}")

                    new_obj: Dict[str, Any] = dict(obj)

                    # ✅写入来源信息（这里不会覆盖，因为上面第一遍已做冲突检查）
                    new_obj[source_name_field] = name
                    new_obj[source_path_field] = str(p)

                    # ✅只改 id：source_name + "__" + 原id
                    if "id" not in new_obj:
                        raise KeyError(f"文件 {p} 第 {lineno} 行缺少字段 'id'，无法拼接 source_name+id")
                    old_id = str(new_obj["id"])
                    new_obj["id"] = f"{name}{id_joiner}{old_id}"

                    # 缺失字段补 None
                    for k in global_keys:
                        if k not in new_obj:
                            new_obj[k] = None

                    out_f.write(json.dumps(new_obj, ensure_ascii=False) + "\n")
                    written_total += 1
                    per_file_written[str(p)] += 1

    print("[MERGE] per-file written lines:")
    for name, p in items:
        print(f"  - {name} ({p}): {per_file_written[str(p)]}")
    print(f"[MERGE] total wrote {written_total} lines -> {out_path}")

    return str(out_path)


In [ ]:
work_dir = "original-layout/Projects/Jupyter/deepseek-prover-V2/runs"

Minif2f_real_sketch = work_dir + "/MiniF2F_eval/results/newdata/real_sketch.jsonl"

FormalMATH_real_sketch_0 = work_dir + "/FormalMATH_eval_results/s0-20251204-144105/real_sketch.jsonl"
FormalMATH_real_sketch_1 = work_dir + "/FormalMATH_eval_results/s1-20251204-144207/real_sketch.jsonl"
FormalMATH_real_sketch_2 = work_dir + "/FormalMATH_eval_results/s2-20260103-235415/real_sketch.jsonl"



In [ ]:
list2 = [("Minif2f_real_sketch", Minif2f_real_sketch), 
         ("FormalMATH_real_sketch_0", FormalMATH_real_sketch_0),
         ("FormalMATH_real_sketch_1", FormalMATH_real_sketch_1),
         ("FormalMATH_real_sketch_2", FormalMATH_real_sketch_2)]

merged_path = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/merged_real_sketch.jsonl"

merged_path = merge_jsonl_files(list2, merged_path, "source_flie")

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path


with jsonlines.open(merged_path, "r") as reader:

    for i, da in enumerate(reader, start=1):
        # dataid = (da.get("parent_id")).strip()
        # stepid = da.get("id")
        # num_steps = da.get("num_steps")
        # theorem = da.get("theorem")
        # sketch_state = da.get("sketch_state")
        # target = da.get("target")
        for k, v in da.items():
            print(f"\n===== {k} =====")
            if isinstance(v, str):
                print(v)      # 这里会显示真实换行
            else:
                print(v)

        
        if i >= 1:
            break

# 加入“source_bench”字段

## 提取namelist

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path

MiniF2F_namelist = []
with jsonlines.open(Minif2f_real_sketch, "r") as reader:

    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        MiniF2F_namelist.append(name)
        

print(len(MiniF2F_namelist))

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List


def add_source_bench_by_name(
    jsonl_path: str,
    name_list: List[str],
    bench_name: str,
    out_path: str,
    other: str = "other",
) -> str:
    """
    一遍处理（不扫描）：
      - name 在 name_list 中：强制 source_bench = bench_name
      - name 不在 name_list 中：
          * 若 source_bench 不存在或为 null -> 写 other
          * 若 source_bench 非 null -> 保留不动
      - 其他字段不变
      - 输出保证每一行都有 source_bench
    """
    src = Path(jsonl_path)
    if not src.exists():
        raise FileNotFoundError(f"输入文件不存在: {src}")

    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    if not isinstance(name_list, (list, tuple)):
        raise TypeError("name_list 必须是 list/tuple")    
    
    name_set = set(str(x) for x in name_list)

    written = 0
    with src.open("r", encoding="utf-8") as fin, out.open("w", encoding="utf-8") as fout:
        for lineno, line in enumerate(fin, start=1):
            raw = line.strip()
            if not raw:
                continue

            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON解析失败: file={src} line={lineno} err={e}") from e
            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={src} line={lineno}")

            name = obj.get("name", None)

            if isinstance(name, str) and name in name_set:
                obj["source_bench"] = bench_name
            else:
                # 不在 list：缺失或 null -> other；非 null -> 保留
                if ("source_bench" not in obj) or (obj["source_bench"] is None):
                    obj["source_bench"] = other

            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            written += 1

    print(f"[OK] wrote {written} lines -> {out}")
    return str(out)


In [ ]:
Final_sketchdata_path = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/Final_sketchdata.jsonl"

add_source_bench_by_name(merged_path, MiniF2F_namelist, "MiniF2F", Final_sketchdata_path, other = "FormalMATH")
                        

### 结果检测

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path

Formal_namelist_0 = []
with jsonlines.open(FormalMATH_real_sketch_0, "r") as reader:

    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        Formal_namelist_0.append(name)
        

print(len(Formal_namelist_0))

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path

Formal_namelist_1 = []
with jsonlines.open(FormalMATH_real_sketch_1, "r") as reader:

    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        Formal_namelist_1.append(name)
        

print(len(Formal_namelist_1))

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path

Formal_namelist_2 = []
with jsonlines.open(FormalMATH_real_sketch_2, "r") as reader:

    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        Formal_namelist_2.append(name)
        

print(len(Formal_namelist_2))

In [ ]:
Formal_namelist = Formal_namelist_0 + Formal_namelist_1 + Formal_namelist_2
print(len(Formal_namelist))

In [ ]:
print(len(Formal_namelist)+len(MiniF2F_namelist))

In [ ]:
import re
import json, jsonlines, hashlib, re
from pathlib import Path

testnamelist = []
with jsonlines.open(Final_sketchdata_path, "r") as reader:

    for i, da in enumerate(reader, start=1):
        name = da.get("name")
        source_bench = da.get("source_bench")
        if source_bench == "FormalMATH" :
            testnamelist.append(name)

print(len(testnamelist))        

In [ ]:
testnamelist == Formal_namelist

# 查看是否含有相同name字段

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List, Dict, Any, Tuple


def find_duplicate_names_across_jsonl(jsonl_paths: List[str]) -> Dict[str, Any]:
    """
    输入：多个 .jsonl 文件路径 list
    功能：检查不同文件之间 name 字段是否存在相同值（跨文件重复）
    输出：dict 结果（不修改任何文件）

    返回结构：
    {
      "total_files": int,
      "total_rows_scanned": int,
      "files": {path: {"rows": int, "rows_with_name": int, "rows_missing_name": int}},
      "duplicates": {
          "<name_value>": [
              {"file": "...", "count": int, "line_nos": [..]},
              ...
          ],
          ...
      }
    }

    说明：
    - 只统计“跨文件重复”：同一个 name 至少出现在 2 个不同文件中才会被列为 duplicates
    - name==None 或空字符串 "" 会被忽略（不参与重复检测）
    - 每行必须是 JSON object(dict)，否则报错
    """
    if not jsonl_paths:
        raise ValueError("jsonl_paths 不能为空")

    paths = [Path(p) for p in jsonl_paths]
    for p in paths:
        if not p.exists():
            raise FileNotFoundError(f"输入文件不存在: {p}")

    # name -> file -> {"count":..., "line_nos":[...]}
    name_map: Dict[str, Dict[str, Dict[str, Any]]] = {}

    file_stats: Dict[str, Dict[str, int]] = {}
    total_rows_scanned = 0

    for p in paths:
        p_str = str(p)
        rows = 0
        rows_with_name = 0
        rows_missing_name = 0

        with p.open("r", encoding="utf-8") as f:
            for lineno, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                rows += 1
                total_rows_scanned += 1

                try:
                    obj = json.loads(line)
                except Exception as e:
                    raise ValueError(f"JSON 解析失败: file={p} line={lineno} err={e}") from e
                if not isinstance(obj, dict):
                    raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno} got={type(obj)}")

                if "name" not in obj:
                    rows_missing_name += 1
                    continue

                name_val = obj.get("name")
                # 忽略 None / ""（你也可以改成要统计）
                if name_val is None or name_val == "":
                    rows_missing_name += 1
                    continue

                # 强制转成 str（避免数字/其他类型导致同值不同型）
                name_val = str(name_val)
                rows_with_name += 1

                if name_val not in name_map:
                    name_map[name_val] = {}
                if p_str not in name_map[name_val]:
                    name_map[name_val][p_str] = {"count": 0, "line_nos": []}

                name_map[name_val][p_str]["count"] += 1
                name_map[name_val][p_str]["line_nos"].append(lineno)

        file_stats[p_str] = {
            "rows": rows,
            "rows_with_name": rows_with_name,
            "rows_missing_name": rows_missing_name,
        }

    # 只保留跨文件重复：出现于 >=2 个不同文件
    duplicates: Dict[str, List[Dict[str, Any]]] = {}
    for name_val, per_file in name_map.items():
        if len(per_file) >= 2:
            duplicates[name_val] = [
                {"file": fp, "count": info["count"], "line_nos": info["line_nos"]}
                for fp, info in sorted(per_file.items(), key=lambda x: x[0])
            ]

    result = {
        "total_files": len(paths),
        "total_rows_scanned": total_rows_scanned,
        "files": file_stats,
        "n_duplicate_names": len(duplicates),
        "duplicates": duplicates,
    }
    
    # 重复的名字list
    duplicate_name_list = list(result["duplicates"].keys())
    
    print("\n")
    print("duplicate_name_list:", duplicate_name_list)
    
    
    return result


# 如果只需要重复的名字的list，可以使用：
# result = find_duplicate_names_across_jsonl(  )
# duplicate_name_list = list(result["duplicates"].keys())

In [ ]:
test = "original-layout/Projects/Jupyter/SketchFill/stepwise_training-4/data/sketch.raw.test.jsonl"
train = "original-layout/Projects/Jupyter/SketchFill/stepwise_training-4/data/sketch.raw.train.jsonl"
valid = "original-layout/Projects/Jupyter/SketchFill/stepwise_training-4/data/sketch.raw.valid.jsonl"

filelist = [test, train, valid]

find_duplicate_names_across_jsonl(filelist)

In [ ]:

verify_n = "original-layout/Projects/Jupyter/SketchFill/stepwise_training-3/test_runs/20260113-122607/summary/verify_n.jsonl"

x = [Minif2f_real_sketch, verify_n]
result = find_duplicate_names_across_jsonl(x)

In [ ]:
duplicate_name_list = list(result["duplicates"].keys())

print(duplicate_name_list)

In [ ]:
namelist1 = ['mathd_numbertheory_780', 'numbertheory_sqmod4in01d', 'amc12a_2009_5', 
             'algebra_amgm_prod1toneq1_sum1tongeqn', 'mathd_numbertheory_257', 
             'mathd_algebra_547', 'mathd_numbertheory_200', 'mathd_algebra_28', 
             'mathd_numbertheory_543', 'mathd_algebra_393', 'mathd_algebra_15', 
             'aime1991_1', 'mathd_numbertheory_284', 'mathd_numbertheory_22',
             'mathd_algebra_149', 'algebra_sqineq_36azm9asqle36zsq',
             'numbertheory_prmdvsneqnsqmodpeq0', 'amc12a_2016_3',
             'algebra_sqineq_2unitcircatblt1', 'mathd_algebra_10', 'amc12b_2020_2',
             'numbertheory_x5neqy2p4', 'induction_12dvd4expnp1p20', 'mathd_numbertheory_328',
             'aime1997_9', 'mathd_algebra_188', 'mathd_algebra_346', 'mathd_numbertheory_769',
             'amc12b_2002_7', 'mathd_algebra_80', 'mathd_algebra_598', 'induction_1pxpownlt1pnx',
             'induction_11div10tonmn1ton', 'mathd_algebra_196', 'amc12a_2021_3', 'mathd_numbertheory_235',
             'mathd_algebra_142', 'mathd_algebra_441', 'mathd_numbertheory_582']

In [ ]:
duplicate_name_list == namelist1

In [ ]:
x = [Minif2f_real_sketch, train]
result = find_duplicate_names_across_jsonl(x)
duplicate_name_list = list(result["duplicates"].keys())

In [ ]:
print(len(duplicate_name_list ))

In [ ]:

raw_test = "original-layout/Projects/Jupyter/SketchFill/stepwise_training-3/data/sketch.raw.test.jsonl"

x = [Minif2f_real_sketch, raw_test]
result = find_duplicate_names_across_jsonl(x)

duplicate_name_list = list(result["duplicates"].keys())


In [ ]:
print(duplicate_name_list)

In [ ]:
duplicate_name_list == namelist1

# 查看有几种不同的name

In [ ]:
import json
from pathlib import Path
from typing import Dict, Any, Optional


def count_unique_names(
    jsonl_path: str,
    name_field: str = "name",
    group_field: str = "source_bench",
    strict: bool = False,
) -> Dict[str, Any]:
    """
    输入一个 .jsonl 文件，统计不同 name 的数量：
      - ALL：全体 unique name 数
      - BY_GROUP：按 source_bench 分组的 unique name 数

    strict=False：
      - 遇到缺失/空 name 的行 -> 计入 missing，但不报错
    strict=True：
      - 遇到缺失/空 name 的行 -> 直接报错

    返回 dict 结果，同时会 print 输出汇总。
    """
    p = Path(jsonl_path)
    if not p.exists():
        raise FileNotFoundError(f"输入文件不存在: {p}")

    total_rows = 0
    missing_name_rows = 0

    all_names = set()
    by_group_names: Dict[str, set] = {}

    by_group_rows: Dict[str, int] = {}
    by_group_missing_name_rows: Dict[str, int] = {}

    with p.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            raw = line.rstrip("\n")
            if not raw.strip():
                continue

            total_rows += 1

            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON 解析失败: file={p} line={lineno} err={e}") from e
            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno}")

            # group: source_bench（允许缺失 -> 归到 "null"）
            g = obj.get(group_field, None)
            g_name = "null" if g is None else str(g)

            by_group_rows[g_name] = by_group_rows.get(g_name, 0) + 1

            # name
            name = obj.get(name_field, None)
            if not isinstance(name, str) or name == "":
                missing_name_rows += 1
                by_group_missing_name_rows[g_name] = by_group_missing_name_rows.get(g_name, 0) + 1
                if strict:
                    raise ValueError(f"第 {lineno} 行 '{name_field}' 缺失或为空")
                continue

            all_names.add(name)

            if g_name not in by_group_names:
                by_group_names[g_name] = set()
            by_group_names[g_name].add(name)

    result = {
        "file": str(p),
        "total_rows_scanned": total_rows,
        "missing_name_rows": missing_name_rows,
        "ALL": {
            "unique_name_count": len(all_names),
        },
        "BY_GROUP": {},
    }

    # 填充分组结果
    for g_name in sorted(by_group_rows.keys()):
        s = by_group_names.get(g_name, set())
        result["BY_GROUP"][g_name] = {
            "rows": by_group_rows[g_name],
            "missing_name_rows": by_group_missing_name_rows.get(g_name, 0),
            "unique_name_count": len(s),
        }

    # -------- print 汇总 --------
    print("\n[UNIQUE NAME COUNT]")
    print(f"  file: {result['file']}")
    print(f"  total_rows_scanned: {total_rows}")
    print(f"  missing_name_rows : {missing_name_rows}")

    print("\n[ALL]")
    print(f"  unique_name_count: {result['ALL']['unique_name_count']}")

    print("\n[BY source_bench]")
    for g_name, info in result["BY_GROUP"].items():
        print(
            f"  - {g_name}: unique_name_count={info['unique_name_count']}  "
            f"(rows={info['rows']}, missing_name_rows={info['missing_name_rows']})"
        )

    return result


In [ ]:
p = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/Final_sketchdata.jsonl"

stats = count_unique_names(p)
print(stats)

# 查看是否有相同的id

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, Any, List


def check_duplicate_ids_in_jsonl(jsonl_path: str) -> Dict[str, Any]:
    """
    输入：一个 .jsonl 文件路径
    功能：检查每行数据中的 "id" 字段是否有重复值
    输出：统计结果 dict，并在函数内打印摘要

    返回的 dict 包含：
      - total_rows_scanned: 扫描的有效行数（非空行）
      - rows_with_id: 有 id 字段的行数
      - rows_missing_id: 缺少 id 字段的行数
      - n_unique_ids: 唯一 id 个数
      - n_duplicate_ids: 重复 id 的种类数（比如 A 重复、B 重复 -> 2）
      - duplicates: {id_value: {"count": n, "line_nos": [...]}, ...}
    """
    path = Path(jsonl_path)
    if not path.exists():
        raise FileNotFoundError(f"输入文件不存在: {path}")

    id2lines: Dict[str, List[int]] = {}
    total_rows = 0
    rows_with_id = 0
    rows_missing_id = 0

    with path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue

            total_rows += 1

            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON解析失败: file={path} line={lineno} err={e}") from e

            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={path} line={lineno}")

            if "id" not in obj:
                rows_missing_id += 1
                continue

            rows_with_id += 1
            id_val = str(obj["id"])  # 统一转成字符串比较
            id2lines.setdefault(id_val, []).append(lineno)

    duplicates = {
        idv: {"count": len(lines), "line_nos": lines}
        for idv, lines in id2lines.items()
        if len(lines) > 1
    }

    result = {
        "file": str(path),
        "total_rows_scanned": total_rows,
        "rows_with_id": rows_with_id,
        "rows_missing_id": rows_missing_id,
        "n_unique_ids": len(id2lines),
        "n_duplicate_ids": len(duplicates),
        "duplicates": duplicates,
    }

    # 打印摘要
    print("\n[DUPLICATE ID CHECK]")
    print(f"  file: {path}")
    print(f"  total_rows_scanned: {total_rows}")
    print(f"  rows_with_id: {rows_with_id}")
    print(f"  rows_missing_id: {rows_missing_id}")
    print(f"  unique_ids: {len(id2lines)}")
    print(f"  duplicate_id_kinds: {len(duplicates)}")

    # 如果有重复，额外打印前几个例子
    if duplicates:
        print("\n  examples (first 10 duplicate ids):")
        for i, (idv, info) in enumerate(duplicates.items()):
            if i >= 10:
                break
            print(f"    - id={idv}  count={info['count']}  lines={info['line_nos'][:20]}"
                  + (" ..." if len(info['line_nos']) > 20 else ""))

    return result


# 用法示例：
# res = check_duplicate_ids_in_jsonl("merged.jsonl")
# print(res["n_duplicate_ids"])


# 查看sketch中洞的数量

In [ ]:
import json
from pathlib import Path
from typing import Dict, Any, Optional


def analyze_proof_token_counts(
    jsonl_path: str,
    token: str = "<proof>",
    sketch_field: str = "sketch",
    group_field: str = "source_bench",
    id_field: str = "id",
    strict: bool = True,
) -> Dict[str, Any]:
    """
    输入一个 .jsonl 文件，统计 sketch 中 <proof> 记号数量：
      - ALL 全体统计
      - 按 source_bench 分组统计

    对每个范围记录：
      - max / min / avg
      - 以及对应样本 id（min_id / max_id）

    strict=True 时：
      - 某一行没有 sketch 字段 或 sketch 为空字符串 -> 直接报错
    strict=False 时：
      - 遇到 sketch 缺失/非法 -> 跳过该行，并记录 skipped 数量

    返回：
      dict 结构统计结果，同时会 print 输出汇总。
    """
    p = Path(jsonl_path)
    if not p.exists():
        raise FileNotFoundError(f"输入文件不存在: {p}")

    def _init_stat():
        return {
            "n": 0,
            "sum": 0,
            "min": None,
            "max": None,
            "min_id": None,
            "max_id": None,
        }

    def _update_stat(stat: Dict[str, Any], cnt: int, rid: Optional[str]):
        stat["n"] += 1
        stat["sum"] += cnt

        if stat["min"] is None or cnt < stat["min"]:
            stat["min"] = cnt
            stat["min_id"] = rid
        if stat["max"] is None or cnt > stat["max"]:
            stat["max"] = cnt
            stat["max_id"] = rid

    overall = _init_stat()
    by_group: Dict[str, Dict[str, Any]] = {}
    skipped = 0
    scanned = 0

    with p.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            raw = line.rstrip("\n")
            if not raw.strip():
                continue

            scanned += 1

            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON 解析失败: file={p} line={lineno} err={e}") from e

            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno}")

            # --- 读取 sketch ---
            if sketch_field not in obj:
                if strict:
                    raise KeyError(f"文件中第 {lineno} 行没有 '{sketch_field}' 字段")
                skipped += 1
                continue

            sketch = obj.get(sketch_field)
            if not isinstance(sketch, str) or sketch == "":
                if strict:
                    raise ValueError(f"文件中第 {lineno} 行 '{sketch_field}' 字段为空或不是字符串")
                skipped += 1
                continue

            # --- 计数 ---
            cnt = sketch.count(token)

            # --- id / group ---
            rid = obj.get(id_field)
            rid = str(rid) if rid is not None else None

            g = obj.get(group_field, None)
            # json null 会被读成 None，这里统一显示成 "null"
            g_name = "null" if g is None else str(g)

            # --- 更新统计 ---
            _update_stat(overall, cnt, rid)

            if g_name not in by_group:
                by_group[g_name] = _init_stat()
            _update_stat(by_group[g_name], cnt, rid)

    def _finalize(stat: Dict[str, Any]) -> Dict[str, Any]:
        n = stat["n"]
        avg = (stat["sum"] / n) if n > 0 else None
        return {
            "n": n,
            "avg": avg,
            "min": stat["min"],
            "min_id": stat["min_id"],
            "max": stat["max"],
            "max_id": stat["max_id"],
        }

    result = {
        "file": str(p),
        "token": token,
        "sketch_field": sketch_field,
        "group_field": group_field,
        "scanned_rows": scanned,
        "skipped_rows": skipped,
        "ALL": _finalize(overall),
        "BY_GROUP": {k: _finalize(v) for k, v in by_group.items()},
    }

    # -------- 打印输出 --------
    print("\n[PROOF TOKEN COUNT ANALYSIS]")
    print(f"  file: {result['file']}")
    print(f"  token: {token}")
    print(f"  scanned_rows: {scanned}  skipped_rows: {skipped}")

    all_st = result["ALL"]
    print("\n[ALL]")
    print(f"  n={all_st['n']}  avg={all_st['avg']}")
    print(f"  min={all_st['min']}  min_id={all_st['min_id']}")
    print(f"  max={all_st['max']}  max_id={all_st['max_id']}")

    print("\n[BY source_bench]")
    for k in sorted(result["BY_GROUP"].keys()):
        st = result["BY_GROUP"][k]
        print(f"  - {k}: n={st['n']}  avg={st['avg']}  min={st['min']} (id={st['min_id']})  max={st['max']} (id={st['max_id']})")

    return result


In [ ]:

p = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/Final_sketchdata.jsonl"

stats = analyze_proof_token_counts(p)
print(stats)

# 统计fills字段的行数

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Dict, Any, Optional


def count_fill_element_lines_by_source_bench(
    jsonl_path: str,
    fills_field: str = "fills",
    bench_field: str = "source_bench",
) -> Dict[Optional[str], Dict[str, Any]]:
    """
    统计：按 source_bench 分组，fills(list) 中每个元素(字符串)的“行数”并汇总。

    行数定义：
      - 先统一换行符：\r\n / \r -> \n
      - lines = s.split("\n")
      - 若 s == "" 则按 1 行计（避免 0 行）

    返回：
      stats[bench] = {
        "n_rows_scanned": 扫描到的行数（JSON object 行）,
        "n_rows_with_fills": fills 字段存在且为 list 的行数,
        "n_fill_elements": fills 中元素总个数,
        "total_lines": 所有 fill 元素的行数总和,
        "avg_lines_per_element": 平均每个元素多少行（若无元素则 None）,
        "min_lines": 最小行数（若无元素则 None）,
        "max_lines": 最大行数（若无元素则 None）,
        "line_count_hist": {行数: 个数, ...}  # 例如 {1: 120, 2: 35}
      }
    """
    p = Path(jsonl_path)
    if not p.exists():
        raise FileNotFoundError(f"输入文件不存在: {p}")

    stats: Dict[Optional[str], Dict[str, Any]] = {}

    def _get_bucket(bench: Optional[str]) -> Dict[str, Any]:
        if bench not in stats:
            stats[bench] = {
                "n_rows_scanned": 0,
                "n_rows_with_fills": 0,
                "n_fill_elements": 0,
                "total_lines": 0,
                "avg_lines_per_element": None,
                "min_lines": None,
                "max_lines": None,
                "line_count_hist": {},  # int -> int
            }
        return stats[bench]

    def _count_lines(s: str) -> int:
        s2 = s.replace("\r\n", "\n").replace("\r", "\n")
        # 空串也算 1 行
        if s2 == "":
            return 1
        return len(s2.split("\n"))

    with p.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON解析失败: file={p} line={lineno} err={e}") from e
            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno}")

            bench = obj.get(bench_field, None)
            if bench is not None and not isinstance(bench, str):
                bench = str(bench)

            bucket = _get_bucket(bench)
            bucket["n_rows_scanned"] += 1

            fills = obj.get(fills_field, None)
            if not isinstance(fills, list):
                continue
            bucket["n_rows_with_fills"] += 1

            for x in fills:
                # 非字符串也转成字符串统计（你也可改为直接跳过）
                if x is None:
                    s = ""
                elif isinstance(x, str):
                    s = x
                else:
                    s = str(x)

                nlines = _count_lines(s)

                bucket["n_fill_elements"] += 1
                bucket["total_lines"] += nlines

                if bucket["min_lines"] is None:
                    bucket["min_lines"] = nlines
                    bucket["max_lines"] = nlines
                else:
                    bucket["min_lines"] = min(bucket["min_lines"], nlines)
                    bucket["max_lines"] = max(bucket["max_lines"], nlines)

                hist = bucket["line_count_hist"]
                hist[nlines] = hist.get(nlines, 0) + 1

    # finalize avg
    for bench, bucket in stats.items():
        n = bucket["n_fill_elements"]
        bucket["avg_lines_per_element"] = (bucket["total_lines"] / n) if n else None

    # print summary
    print("[FILL LINE COUNT BY source_bench]")
    for bench, bucket in stats.items():
        print(f"\n- source_bench={bench}")
        print(f"  rows_scanned        = {bucket['n_rows_scanned']}")
        print(f"  rows_with_fills     = {bucket['n_rows_with_fills']}")
        print(f"  fill_elements       = {bucket['n_fill_elements']}")
        print(f"  total_lines         = {bucket['total_lines']}")
        print(f"  avg_lines/element   = {bucket['avg_lines_per_element']}")
        print(f"  min_lines           = {bucket['min_lines']}")
        print(f"  max_lines           = {bucket['max_lines']}")
        # 只打印前若干个行数分布（避免太长）
        hist_items = sorted(bucket["line_count_hist"].items())[:20]
        print(f"  line_count_hist(top)= {hist_items} ...")

    return stats


In [ ]:

sketch_path = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/Final_sketchdata.jsonl"

In [ ]:
stats = count_fill_element_lines_by_source_bench(sketch_path , "fills")

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Dict, Any, Optional


def count_fill_element_lines_by_source_bench(
    jsonl_path: str,
    fills_field: str = "fills",
    bench_field: str = "source_bench",
    id_field: str = "id",
    all_key: str = "__ALL__",   # ✅总体桶的 key（避免和真实 bench 名冲突）
) -> Dict[Optional[str], Dict[str, Any]]:
    """
    统计：按 source_bench 分组 + 总体(all_key) 统计，
    fills(list) 中每个元素(字符串)的“行数”并汇总。
    ✅记录：最大行数对应的样本 id 列表（max_lines_ids）
    """
    p = Path(jsonl_path)
    if not p.exists():
        raise FileNotFoundError(f"输入文件不存在: {p}")

    stats: Dict[Optional[str], Dict[str, Any]] = {}

    def _get_bucket(bench: Optional[str]) -> Dict[str, Any]:
        if bench not in stats:
            stats[bench] = {
                "n_rows_scanned": 0,
                "n_rows_with_fills": 0,
                "n_fill_elements": 0,
                "total_lines": 0,
                "avg_lines_per_element": None,
                "min_lines": None,
                "max_lines": None,
                "max_lines_ids": [],          # ✅新增
                "_max_lines_id_set": set(),   # 内部去重
                "line_count_hist": {},        # int -> int
            }
        return stats[bench]

    def _count_lines(s: str) -> int:
        s2 = s.replace("\r\n", "\n").replace("\r", "\n")
        # 空串按 1 行计
        return 1 if s2 == "" else len(s2.split("\n"))

    def _update_bucket(bucket: Dict[str, Any], row_id: Optional[str], nlines: int):
        bucket["n_fill_elements"] += 1
        bucket["total_lines"] += nlines

        # min/max
        if bucket["min_lines"] is None:
            bucket["min_lines"] = nlines
            bucket["max_lines"] = nlines
            bucket["max_lines_ids"] = []
            bucket["_max_lines_id_set"] = set()
            if row_id is not None:
                bucket["max_lines_ids"].append(row_id)
                bucket["_max_lines_id_set"].add(row_id)
        else:
            bucket["min_lines"] = min(bucket["min_lines"], nlines)

            cur_max = bucket["max_lines"]
            if nlines > cur_max:
                bucket["max_lines"] = nlines
                bucket["max_lines_ids"] = []
                bucket["_max_lines_id_set"] = set()
                if row_id is not None:
                    bucket["max_lines_ids"].append(row_id)
                    bucket["_max_lines_id_set"].add(row_id)
            elif nlines == cur_max:
                if row_id is not None and row_id not in bucket["_max_lines_id_set"]:
                    bucket["max_lines_ids"].append(row_id)
                    bucket["_max_lines_id_set"].add(row_id)

        # hist
        hist = bucket["line_count_hist"]
        hist[nlines] = hist.get(nlines, 0) + 1

    with p.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON解析失败: file={p} line={lineno} err={e}") from e
            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno}")

            bench = obj.get(bench_field, None)
            if bench is not None and not isinstance(bench, str):
                bench = str(bench)

            # ✅两个桶：该 bench + 总体
            bucket_bench = _get_bucket(bench)
            bucket_all = _get_bucket(all_key)

            bucket_bench["n_rows_scanned"] += 1
            bucket_all["n_rows_scanned"] += 1

            fills = obj.get(fills_field, None)
            if not isinstance(fills, list):
                continue

            bucket_bench["n_rows_with_fills"] += 1
            bucket_all["n_rows_with_fills"] += 1

            row_id = obj.get(id_field, None)
            row_id = str(row_id) if row_id is not None else None

            for x in fills:
                if x is None:
                    s = ""
                elif isinstance(x, str):
                    s = x
                else:
                    s = str(x)

                nlines = _count_lines(s)

                _update_bucket(bucket_bench, row_id, nlines)
                _update_bucket(bucket_all, row_id, nlines)

    # finalize avg + 清理内部字段
    for _, bucket in stats.items():
        n = bucket["n_fill_elements"]
        bucket["avg_lines_per_element"] = (bucket["total_lines"] / n) if n else None
        bucket.pop("_max_lines_id_set", None)

    # print summary（先打印总体）
    print("[FILL LINE COUNT BY source_bench]")
    if all_key in stats:
        b = stats[all_key]
        print(f"\n- source_bench={all_key} (OVERALL)")
        print(f"  fill_elements       = {b['n_fill_elements']}")
        print(f"  total_lines         = {b['total_lines']}")
        print(f"  avg_lines/element   = {b['avg_lines_per_element']}")
        print(f"  min_lines           = {b['min_lines']}")
        print(f"  max_lines           = {b['max_lines']}")
        print(f"  max_lines_ids       = {b['max_lines_ids'][:20]}{' ...' if len(b['max_lines_ids']) > 20 else ''}")

    for bench, bucket in stats.items():
        if bench == all_key:
            continue
        print(f"\n- source_bench={bench}")
        print(f"  fill_elements       = {bucket['n_fill_elements']}")
        print(f"  total_lines         = {bucket['total_lines']}")
        print(f"  avg_lines/element   = {bucket['avg_lines_per_element']}")
        print(f"  min_lines           = {bucket['min_lines']}")
        print(f"  max_lines           = {bucket['max_lines']}")
        print(f"  max_lines_ids       = {bucket['max_lines_ids'][:20]}{' ...' if len(bucket['max_lines_ids']) > 20 else ''}")

    return stats


In [ ]:
stats = count_fill_element_lines_by_source_bench(sketch_path , "fills")

# 数据性质分析
      1) 各 source_bench 的总数据量
      2) 不同 source_bench 间是否存在相同 name（给出交集数量 + 例子）
      3) 不同 source_bench 间是否存在相同 id（给出交集数量 + 例子）
      4) ALL/各bench 内部是否存在相同的 id（duplicate_id_kinds + examples）
      5) ALL + 各 source_bench 层面统计：
         - unique name 数
         - total_holes / multi_hole_rows
         - sketch 中 <proof> 次数：min/max/avg + min_id/max_id
         - fills(list) 长度：min/max/avg + min_id/max_id
         - fills 元素行数：总元素数/总行数/min/max/avg + max_lines 对应 id + 行数直方图
         - holes(<proof> per-row) 的 median / p90
         - fill_lines(per-fill-element) 的 median / p90
         - fill_tokens(per-fill-element) 的 median / p90

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, Any, Optional
from collections import defaultdict

from transformers import AutoTokenizer


def analyze_jsonl_overview_by_source_bench(
    jsonl_path: str,
    bench_field: str = "source_bench",
    name_field: str = "name",
    id_field: str = "id",
    sketch_field: str = "sketch",
    fills_field: str = "fills",
    proof_token: str = "<proof>",
    strict_sketch: bool = False,
    strict_fills: bool = False,
    keep_examples_per_scope_dup: int = 20,
    keep_mismatch_examples: int = 30,
    tokenizer_path: str = "original-layout/Models/DeepSeek-Prover-V2-7B",
) -> Dict[str, Any]:
    """
    输入一个 .jsonl 文件，输出“总览统计”：
      1) 各 source_bench 的总数据量
      2) 不同 source_bench 间是否存在相同 name（给出交集数量 + 例子）
      3) 不同 source_bench 间是否存在相同 id（给出交集数量 + 例子）
      4) ALL + 各 source_bench 层面统计：
         - unique name 数
         - sketch 中 <proof> 次数：min/max/avg + min_id/max_id
         - fills(list) 长度：min/max/avg + min_id/max_id
         - fills 元素行数：总元素数/总行数/min/max/avg + max_lines 对应 id + 行数直方图

    口径：
      - source_bench 缺失/None -> "null"
      - name 缺失/空串：计入 missing，但不计入 unique_name（不报错）
      - id 缺失：计入 missing，但不计入重复检测
      - sketch：strict_sketch=True 时缺失/空串报错；否 则跳过该行 sketch 统计并计入 skipped
      - fills：strict_fills=True 时缺失/非 list 报错；否 则跳过 fills 统计并计入 skipped
      - fills 行数：先统一换行符；"" 计 1 行；否则 len(split("\n"))

    改动点（其他逻辑不变）：
      1) 将 sketch_token 改为 洞数（hole_count），并额外统计 total_holes / multi_hole_rows
      2) 增加：ALL/各bench 内部是否存在相同的 id（duplicate_id_kinds + examples）
      3) print 输出格式：等号两边加空格
    （本次）：对每个 scope(ALL/bench) 增加 quantile：
      - holes(<proof> per-row) 的 median / p90
      - fill_lines(per-fill-element) 的 median / p90
      - fill_tokens(per-fill-element) 的 median / p90
    """
    p = Path(jsonl_path)
    if not p.exists():
        raise FileNotFoundError(f"输入文件不存在: {p}")

    # tokenizer 只加载一次
    tok = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True)

    def _bench_name(v) -> str:
        if v is None:
            return "null"
        return v if isinstance(v, str) else str(v)

    def _normalize_newlines_keep_ws(s: str) -> str:
        return s.replace("\r\n", "\n").replace("\r", "\n")

    def _count_lines(s: str) -> int:
        s2 = _normalize_newlines_keep_ws(s)
        if s2 == "":
            return 1
        return len(s2.split("\n"))

    def _count_tokens(s: str) -> int:
        return len(tok.encode(s, add_special_tokens=False))

    # ---- quantile helpers（线性插值，类似 numpy.quantile(method="linear")）----
    def _quantile_linear(sorted_vals: list[int], q: float) -> Optional[float]:
        if not sorted_vals:
            return None
        n = len(sorted_vals)
        if n == 1:
            return float(sorted_vals[0])
        if q <= 0:
            return float(sorted_vals[0])
        if q >= 1:
            return float(sorted_vals[-1])

        pos = q * (n - 1)
        lo = int(pos)
        hi = min(lo + 1, n - 1)
        frac = pos - lo
        return (1.0 - frac) * float(sorted_vals[lo]) + frac * float(sorted_vals[hi])

    def _median_p90(vals: list[int]) -> tuple[Optional[float], Optional[float]]:
        if not vals:
            return None, None
        s = sorted(vals)
        return _quantile_linear(s, 0.5), _quantile_linear(s, 0.9)

    def _init_group_stat() -> Dict[str, Any]:
        return {
            "rows_scanned": 0,
            "rows_missing_name": 0,
            "rows_missing_id": 0,
            "unique_names": set(),

            # holes
            "hole_rows_used": 0,
            "hole_rows_skipped": 0,
            "hole_sum": 0,
            "hole_min": None,
            "hole_max": None,
            "hole_min_id": None,
            "hole_max_id": None,
            "total_holes": 0,
            "multi_hole_rows": 0,
            # ✅新增：为 median/p90 保存每行洞数
            "hole_values": [],  # list[int]

            # fills length
            "fills_rows_used": 0,
            "fills_rows_skipped": 0,
            "fills_len_sum": 0,
            "fills_len_min": None,
            "fills_len_max": None,
            "fills_len_min_id": None,
            "fills_len_max_id": None,

            # fill lines (element-level)
            "fill_elements": 0,
            "total_lines": 0,
            "min_lines": None,
            "max_lines": None,
            "max_lines_id": None,
            "avg_lines_per_element": None,
            "line_count_hist": {},
            # ✅新增：为 median/p90 保存每个元素行数
            "fill_line_values": [],  # list[int]

            # fill tokens (element-level)
            "fill_token_elements": 0,
            "fill_token_total": 0,
            "fill_token_min": None,
            "fill_token_max": None,
            "fill_token_min_id": None,
            "fill_token_max_id": None,
            "fill_token_avg": None,
            # ✅新增：为 median/p90 保存每个元素 token 数
            "fill_token_values": [],  # list[int]

            # mismatch: holes vs fills_len
            "mismatch_rows": 0,
            "mismatch_examples": [],

            # scope内部重复 id
            "id_rows_with_id": 0,
            "id_count_map": defaultdict(int),
        }

    def _update_minmax(stat: Dict[str, Any], vmin_k: str, vmax_k: str, vmin_id_k: str, vmax_id_k: str, val: int, rid: Optional[str]):
        if stat[vmin_k] is None or val < stat[vmin_k]:
            stat[vmin_k] = val
            stat[vmin_id_k] = rid
        if stat[vmax_k] is None or val > stat[vmax_k]:
            stat[vmax_k] = val
            stat[vmax_id_k] = rid

    groups: Dict[str, Dict[str, Any]] = {"__ALL__": _init_group_stat()}

    name_to_benches: Dict[str, set] = defaultdict(set)
    id_to_benches: Dict[str, set] = defaultdict(set)
    bench_row_counts: Dict[str, int] = defaultdict(int)

    total_rows_scanned = 0

    with p.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            raw = line.rstrip("\n")
            if not raw.strip():
                continue

            total_rows_scanned += 1

            try:
                obj = json.loads(raw)
            except Exception as e:
                raise ValueError(f"JSON 解析失败: file={p} line={lineno} err={e}") from e
            if not isinstance(obj, dict):
                raise ValueError(f"每行必须是 JSON object(dict): file={p} line={lineno}")

            bench = _bench_name(obj.get(bench_field, None))
            bench_row_counts[bench] += 1
            if bench not in groups:
                groups[bench] = _init_group_stat()

            # rows_scanned
            for b in ("__ALL__", bench):
                groups[b]["rows_scanned"] += 1

            # name / id
            name = obj.get(name_field, None)
            rid = obj.get(id_field, None)
            rid_str = str(rid) if rid is not None else None

            if isinstance(name, str) and name != "":
                for b in ("__ALL__", bench):
                    groups[b]["unique_names"].add(name)
                name_to_benches[name].add(bench)
            else:
                for b in ("__ALL__", bench):
                    groups[b]["rows_missing_name"] += 1

            if rid_str is not None and rid_str != "":
                id_to_benches[rid_str].add(bench)
                for b in ("__ALL__", bench):
                    groups[b]["id_rows_with_id"] += 1
                    groups[b]["id_count_map"][rid_str] += 1
            else:
                for b in ("__ALL__", bench):
                    groups[b]["rows_missing_id"] += 1

            # holes from sketch
            sketch = obj.get(sketch_field, None)
            hole_cnt: Optional[int] = None
            if not isinstance(sketch, str) or sketch == "":
                if strict_sketch:
                    raise ValueError(f"第 {lineno} 行 '{sketch_field}' 缺失/为空")
                for b in ("__ALL__", bench):
                    groups[b]["hole_rows_skipped"] += 1
            else:
                hole_cnt = sketch.count(proof_token)
                for b in ("__ALL__", bench):
                    st = groups[b]
                    st["hole_rows_used"] += 1
                    st["hole_sum"] += hole_cnt
                    st["total_holes"] += hole_cnt
                    if hole_cnt > 1:
                        st["multi_hole_rows"] += 1
                    st["hole_values"].append(int(hole_cnt))  # ✅用于 median/p90
                    _update_minmax(st, "hole_min", "hole_max", "hole_min_id", "hole_max_id", hole_cnt, rid_str)

            # fills(list)
            fills = obj.get(fills_field, None)
            fills_len: Optional[int] = None

            if not isinstance(fills, list):
                if strict_fills:
                    raise ValueError(f"第 {lineno} 行 '{fills_field}' 缺失或不是 list")
                for b in ("__ALL__", bench):
                    groups[b]["fills_rows_skipped"] += 1
            else:
                fills_len = len(fills)
                for b in ("__ALL__", bench):
                    st = groups[b]
                    st["fills_rows_used"] += 1
                    st["fills_len_sum"] += fills_len
                    _update_minmax(st, "fills_len_min", "fills_len_max", "fills_len_min_id", "fills_len_max_id", fills_len, rid_str)

                for x in fills:
                    if x is None:
                        s = ""
                    elif isinstance(x, str):
                        s = x
                    else:
                        s = str(x)

                    # lines per element
                    nlines = _count_lines(s)
                    for b in ("__ALL__", bench):
                        st = groups[b]
                        st["fill_elements"] += 1
                        st["total_lines"] += nlines
                        st["fill_line_values"].append(int(nlines))  # ✅用于 median/p90

                        if st["min_lines"] is None:
                            st["min_lines"] = nlines
                            st["max_lines"] = nlines
                            st["max_lines_id"] = rid_str
                        else:
                            if nlines < st["min_lines"]:
                                st["min_lines"] = nlines
                            if nlines > st["max_lines"]:
                                st["max_lines"] = nlines
                                st["max_lines_id"] = rid_str

                        hist = st["line_count_hist"]
                        hist[nlines] = hist.get(nlines, 0) + 1

                    # tokens per element
                    ntok = _count_tokens(s)
                    for b in ("__ALL__", bench):
                        st = groups[b]
                        st["fill_token_elements"] += 1
                        st["fill_token_total"] += ntok
                        st["fill_token_values"].append(int(ntok))  # ✅用于 median/p90
                        _update_minmax(st, "fill_token_min", "fill_token_max", "fill_token_min_id", "fill_token_max_id", ntok, rid_str)

            # mismatch: only if both available
            if hole_cnt is not None and fills_len is not None:
                if hole_cnt != fills_len:
                    for b in ("__ALL__", bench):
                        st = groups[b]
                        st["mismatch_rows"] += 1
                        if len(st["mismatch_examples"]) < keep_mismatch_examples:
                            st["mismatch_examples"].append({"id": rid_str, "hole_cnt": hole_cnt, "fills_len": fills_len})

    benches = sorted(bench_row_counts.keys())

    def _finalize_bucket(st: Dict[str, Any]) -> Dict[str, Any]:
        out = dict(st)
        out["unique_name_count"] = len(out["unique_names"])
        del out["unique_names"]

        out["hole_avg"] = (out["hole_sum"] / out["hole_rows_used"]) if out["hole_rows_used"] > 0 else None
        out["fills_len_avg"] = (out["fills_len_sum"] / out["fills_rows_used"]) if out["fills_rows_used"] > 0 else None
        out["avg_lines_per_element"] = (out["total_lines"] / out["fill_elements"]) if out["fill_elements"] > 0 else None
        out["fill_token_avg"] = (out["fill_token_total"] / out["fill_token_elements"]) if out["fill_token_elements"] > 0 else None

        # ✅新增：median/p90
        holes_med, holes_p90 = _median_p90(out["hole_values"])
        fill_lines_med, fill_lines_p90 = _median_p90(out["fill_line_values"])
        fill_tok_med, fill_tok_p90 = _median_p90(out["fill_token_values"])
        out["holes_median"] = holes_med
        out["holes_p90"] = holes_p90
        out["fill_lines_median"] = fill_lines_med
        out["fill_lines_p90"] = fill_lines_p90
        out["fill_tokens_median"] = fill_tok_med
        out["fill_tokens_p90"] = fill_tok_p90

        # 不需要在最终 json 里保留原始 values（否则很大）
        del out["hole_values"]
        del out["fill_line_values"]
        del out["fill_token_values"]

        # scope内部重复 id
        id_map = out["id_count_map"]
        dup_ids = [k for k, c in id_map.items() if c > 1]
        dup_ids.sort()
        out["unique_id_count"] = len(id_map)
        out["duplicate_id_kinds"] = len(dup_ids)
        out["duplicate_id_examples"] = dup_ids[:keep_examples_per_scope_dup]
        del out["id_count_map"]

        return out

    # cross-bench overlap counts
    n_multi_bench_names = sum(1 for _, bs in name_to_benches.items() if len(bs) >= 2)
    n_multi_bench_ids = sum(1 for _, bs in id_to_benches.items() if len(bs) >= 2)

    result = {
        "file": str(p),
        "total_rows_scanned": total_rows_scanned,
        "bench_row_counts": dict(sorted(bench_row_counts.items(), key=lambda x: x[0])),
        "name_overlap_across_benches": {"n_multi_bench_names": n_multi_bench_names},
        "id_overlap_across_benches": {"n_multi_bench_ids": n_multi_bench_ids},
        "stats": {b: _finalize_bucket(groups[b]) for b in (["__ALL__"] + benches)},
    }

    # ---------------- print：等号两边加空格 ----------------
    print("\n[OVERVIEW]")
    print(f"  file = {result['file']}")
    print(f"  total_rows_scanned = {result['total_rows_scanned']}")

    print("\n[1) bench counts]")
    for b in benches:
        print(f"  - {b}: {result['bench_row_counts'][b]}")

    print("\n[2) name overlap across benches]")
    print(f"  n_multi_bench_names = {result['name_overlap_across_benches']['n_multi_bench_names']}")

    print("\n[3) id overlap across benches]")
    print(f"  n_multi_bench_ids = {result['id_overlap_across_benches']['n_multi_bench_ids']}")

    def _print_bucket(scope: str):
        st = result["stats"][scope]
        print(f"\n[4) stats: {scope}]")
        print(f"  rows_scanned = {st['rows_scanned']}")
        print(f"  missing_name_rows = {st['rows_missing_name']}")
        print(f"  missing_id_rows = {st['rows_missing_id']}")
        print(f"  unique_name_count = {st['unique_name_count']}")

        print(
            "  holes: "
            f"used = {st['hole_rows_used']}  skipped = {st['hole_rows_skipped']}  "
            f"total_holes = {st['total_holes']}  multi_hole_rows = {st['multi_hole_rows']}  "
            f"min = {st['hole_min']} (id = {st['hole_min_id']})  "
            f"max = {st['hole_max']} (id = {st['hole_max_id']})  "
            f"avg = {st['hole_avg']}  "
            f"median = {st['holes_median']}  p90 = {st['holes_p90']}"
        )

        print(
            "  fills_len: "
            f"used = {st['fills_rows_used']}  skipped = {st['fills_rows_skipped']}  "
            f"min = {st['fills_len_min']} (id = {st['fills_len_min_id']})  "
            f"max = {st['fills_len_max']} (id = {st['fills_len_max_id']})  "
            f"avg = {st['fills_len_avg']}"
        )

        print(f"  holes_vs_fills_len_mismatch_rows = {st['mismatch_rows']}")
        if st["mismatch_rows"] > 0:
            print(f"  mismatch_examples = {st['mismatch_examples']}")

        print(
            "  fill_lines: "
            f"fill_elements = {st['fill_elements']}  total_lines = {st['total_lines']}  "
            f"min_lines = {st['min_lines']}  max_lines = {st['max_lines']} (id = {st['max_lines_id']})  "
            f"avg_lines/elem = {st['avg_lines_per_element']}  "
            f"median = {st['fill_lines_median']}  p90 = {st['fill_lines_p90']}"
        )

        print(
            "  fill_tokens: "
            f"elements = {st['fill_token_elements']}  total_tokens = {st['fill_token_total']}  "
            f"min_tokens = {st['fill_token_min']} (id = {st['fill_token_min_id']})  "
            f"max_tokens = {st['fill_token_max']} (id = {st['fill_token_max_id']})  "
            f"avg_tokens/elem = {st['fill_token_avg']}  "
            f"median = {st['fill_tokens_median']}  p90 = {st['fill_tokens_p90']}"
        )

        print(
            "  id_duplicates: "
            f"unique_id_count = {st['unique_id_count']}  "
            f"duplicate_id_kinds = {st['duplicate_id_kinds']}"
        )
        if st["duplicate_id_kinds"] > 0:
            print(f"  duplicate_id_examples = {st['duplicate_id_examples']}")

    _print_bucket("__ALL__")
    for b in benches:
        _print_bucket(b)

    return result


In [ ]:
SKETCH_JSONL = "original-layout/Projects/Jupyter/deepseek-prover-V2/tuning_data/Final_sketchdata.jsonl"
TEST_JSONL = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/data/sketch.raw.test.jsonl"
TRAIN_JSONL = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/data/sketch.raw.train.jsonl"
VALID_JSONL = "original-layout/Projects/Jupyter/SketchFill/holewise_training_2/data/sketch.raw.valid.jsonl"

In [ ]:
analyze_jsonl_overview_by_source_bench(SKETCH_JSONL)

In [ ]:
analyze_jsonl_overview_by_source_bench(TEST_JSONL)